# Pertemuan 11
## Sustainable Logistics

### Kasus

Ekspor kopi dari Lampung menuju pasar Eropa (Pelabuhan Rotterdam, Belanda) menggunakan tiga rute multimoda:
- Truk + Kapal Laut (jalur darat-laut konvensional)
- Kereta Api + Kapal Laut (multimoda darat-laut)
- Kargo Udara (pengiriman ekspres langsung)

---
## 1. Dataset Moda Transportasi

In [1]:
import pandas as pd

moda = pd.DataFrame({
    'moda':          ['Truk + Kapal Laut', 'Kereta Api + Kapal Laut', 'Kargo Udara'],
    'emisi_co2_kg':  [620,                  380,                       2400],   # kg CO2 per ton muatan (Lampung -> Rotterdam)
    'biaya_rp':      [18000000,             15500000,                  42000000], # Rp per ton
    'lead_time_jam': [720,                  600,                       36],     # jam
    'kapasitas_ton': [20,                   500,                       5],
})

moda

,moda,emisi_co2_kg,biaya_rp,lead_time_jam,kapasitas_ton
0,Truk + Kapal Laut,620,18000000,720,20
1,Kereta Api + Kapal Laut,380,15500000,600,500
2,Kargo Udara,2400,42000000,36,5


---
## 2. Sustainability Score

In [2]:
# Normalisasi: skor lebih rendah = lebih berkelanjutan
df = moda.copy()
df['emisi_norm']  = df['emisi_co2_kg'] / df['emisi_co2_kg'].max()
df['biaya_norm']  = df['biaya_rp']     / df['biaya_rp'].max()
df['waktu_norm']  = df['lead_time_jam']/ df['lead_time_jam'].max()

# Bobot: emisi 50%, biaya 30%, waktu 20%
df['sustainability_score'] = (
    df['emisi_norm'] * 0.5 +
    df['biaya_norm'] * 0.3 +
    df['waktu_norm'] * 0.2
).round(3)

df[['moda', 'emisi_co2_kg', 'biaya_rp', 'lead_time_jam', 'sustainability_score']].sort_values('sustainability_score')

,moda,emisi_co2_kg,biaya_rp,lead_time_jam,sustainability_score
1,Kereta Api + Kapal Laut,380,15500000,600,0.357
0,Truk + Kapal Laut,620,18000000,720,0.458
2,Kargo Udara,2400,42000000,36,0.810


---
## 3. AI Sustainability Advisor (Groq)

In [ ]:
from groq import Groq
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

data_str = moda.to_string(index=False)

prompt = f"""
Berikut data moda transportasi ekspor kopi dari Lampung ke Rotterdam, Belanda:

{data_str}

Moda mana yang paling berkelanjutan untuk ekspor kopi dari Lampung ke Rotterdam?
Pertimbangkan emisi karbon, biaya, dan lead time.
Berikan rekomendasi dan alasannya.
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Berdasarkan data yang diberikan, moda transportasi yang paling berkelanjutan untuk ekspor kopi dari Lampung ke Rotterdam dapat dianalisis dengan mempertimbangkan emisi karbon, biaya, dan lead time.

1. **Emisi Karbon (CO2)**: Kereta Api + Kapal Laut memiliki emisi karbon yang paling rendah, yaitu 380 kg per ton.
2. **Biaya**: Kereta Api + Kapal Laut juga memiliki biaya yang relatif rendah, yaitu Rp 15.500.000 per ton.
3. **Lead Time**: Kargo Udara memiliki lead time yang paling singkat, yaitu 36 jam.

Namun, jika kita mempertimbangkan ketiga faktor tersebut secara keseluruhan, maka Kereta Api + Kapal Laut dapat menjadi pilihan yang paling berkelanjutan. Berikut alasannya:

- **Emisi Karbon**: Kereta Api + Kapal Laut menghasilkan emisi karbon yang jauh lebih rendah (380 kg) dibandingkan dengan Truk + Kapal Laut (620 kg) dan terutama Kargo Udara (2.400 kg) yang emisinya hampir 4x lipat.
- **Biaya**: Biaya Kereta Api + Kapal Laut (Rp 15.500.000) merupakan yang termurah di antara ketiganya

---
## 4. Nilai SI: Chatbot Keberlanjutan

Bangun chatbot interaktif yang menjawab pertanyaan seputar keberlanjutan logistik ekspor.

In [4]:
history = []

def chat_sustainability(pertanyaan: str) -> str:
    history.append({"role": "user", "content": pertanyaan})
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Kamu adalah ahli logistik ekspor berkelanjutan. Jawab dalam Bahasa Indonesia."},
            *history
        ]
    )
    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

# Contoh penggunaan
print(chat_sustainability("Bagaimana cara mengurangi emisi karbon dalam ekspor kopi ke Eropa?"))

Sebagai ahli logistik ekspor berkelanjutan, saya memiliki beberapa saran untuk mengurangi emisi karbon dalam ekspor kopi ke Eropa:

1. **Pilih Moda Laut Dibanding Udara**: Kapal kontainer menghasilkan emisi karbon per ton jauh lebih rendah dibandingkan kargo udara, sehingga sebaiknya menjadi pilihan utama untuk pengiriman dalam jumlah besar.
2. **Optimalkan Rute Pelayaran**: Menggunakan rute pelayaran yang lebih efisien dan menghindari transit berlebih dapat mengurangi jarak tempuh dan emisi karbon.
3. **Konsolidasi Muatan Kontainer**: Menggabungkan beberapa pengiriman dalam satu kontainer (LCL/FCL) dapat mengurangi jumlah perjalanan dan emisi per unit produk.
4. **Gunakan Multimoda Darat-Laut**: Mengombinasikan kereta api dan kapal laut untuk mencapai pelabuhan ekspor dapat menekan emisi dibandingkan truk diesel jarak jauh.
5. **Slow Steaming**: Bekerja sama dengan perusahaan pelayaran yang menerapkan slow steaming (mengurangi kecepatan kapal) untuk menurunkan konsumsi bahan bakar dan

In [5]:
print(chat_sustainability("halo"))

Halo! Selamat datang! Apakah ada yang bisa saya bantu seputar logistik ekspor berkelanjutan?
